# 00 — Data Reduction
One-time step. Reads raw CSVs, strips unnecessary columns, aggregates auxiliary tables, builds master parquet.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW  = Path('../Data/Raw')
PROC = Path('../Data/Processed')
PROC.mkdir(exist_ok=True)

## 1. Strip application_train to essentials

In [2]:
df = pd.read_csv(RAW / 'application_train.csv')
print(f"Shape: {df.shape}")
print(f"Default rate: {df['TARGET'].mean():.2%}")

keep_cols = [
    'SK_ID_CURR', 'TARGET',
    'NAME_CONTRACT_TYPE', 'AMT_CREDIT', 'AMT_ANNUITY',
    'AMT_GOODS_PRICE', 'AMT_INCOME_TOTAL',
    'CODE_GENDER', 'DAYS_BIRTH', 'DAYS_EMPLOYED',
    'DAYS_ID_PUBLISH', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
    'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE',
    'OCCUPATION_TYPE', 'ORGANIZATION_TYPE',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'OWN_CAR_AGE',
    'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT',
    'AMT_REQ_CREDIT_BUREAU_YEAR',
    'DEF_30_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE',
    'REGION_RATING_CLIENT', 'REGION_POPULATION_RELATIVE',
    'REG_CITY_NOT_LIVE_CITY', 'REG_CITY_NOT_WORK_CITY',
]

df_slim = df[keep_cols]
df_slim.to_parquet(PROC / 'application_slim.parquet', index=False)
print(f"\nSlim shape: {df_slim.shape}")
del df, df_slim

Shape: (307511, 122)
Default rate: 8.07%

Slim shape: (307511, 34)


## 2. Aggregate bureau.csv

In [3]:
bureau = pd.read_csv(RAW / 'bureau.csv')

bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    bureau_loan_count    = ('SK_ID_BUREAU', 'count'),
    bureau_active_count  = ('CREDIT_ACTIVE', lambda x: (x == 'Active').sum()),
    bureau_total_debt    = ('AMT_CREDIT_SUM_DEBT', 'sum'),
    bureau_total_credit  = ('AMT_CREDIT_SUM', 'sum'),
    bureau_total_overdue = ('AMT_CREDIT_SUM_OVERDUE', 'sum'),
    bureau_max_overdue   = ('AMT_CREDIT_MAX_OVERDUE', 'max'),
    bureau_avg_days      = ('DAYS_CREDIT', 'mean'),
    bureau_cnt_prolong   = ('CNT_CREDIT_PROLONG', 'sum'),
)
bureau_agg['bureau_debt_ratio'] = (
    bureau_agg['bureau_total_debt'] / bureau_agg['bureau_total_credit'].replace(0, pd.NA)
)

bureau_agg.to_parquet(PROC / 'bureau_agg.parquet')
print(f"bureau_agg shape: {bureau_agg.shape}")
del bureau, bureau_agg

bureau_agg shape: (305811, 9)


## 3. Aggregate previous_application.csv

In [4]:
prev = pd.read_csv(RAW / 'previous_application.csv')

# Most recent 3 applications per customer — recency matters most
prev = prev.sort_values(['SK_ID_CURR', 'DAYS_DECISION'], ascending=[True, False])
prev = prev.groupby('SK_ID_CURR').head(3)

prev_agg = prev.groupby('SK_ID_CURR').agg(
    prev_count       = ('SK_ID_PREV', 'count'),
    prev_approved    = ('NAME_CONTRACT_STATUS', lambda x: (x == 'Approved').sum()),
    prev_refused     = ('NAME_CONTRACT_STATUS', lambda x: (x == 'Refused').sum()),
    prev_avg_credit  = ('AMT_CREDIT', 'mean'),
    prev_avg_annuity = ('AMT_ANNUITY', 'mean'),
    prev_days_last   = ('DAYS_DECISION', 'max'),
)
prev_agg['prev_approval_rate'] = prev_agg['prev_approved'] / prev_agg['prev_count']

prev_agg.to_parquet(PROC / 'prev_agg.parquet')
print(f"prev_agg shape: {prev_agg.shape}")
del prev, prev_agg

prev_agg shape: (338857, 7)


## 4. Aggregate installments_payments.csv

In [5]:
inst = pd.read_csv(RAW / 'installments_payments.csv')

inst['payment_diff'] = inst['AMT_PAYMENT'] - inst['AMT_INSTALMENT']
inst['days_late']    = inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']
inst['is_late']      = (inst['days_late'] > 0).astype(int)

# Last 12 months only
inst = inst[inst['DAYS_INSTALMENT'] > -365]

inst_agg = inst.groupby('SK_ID_CURR').agg(
    inst_count      = ('AMT_INSTALMENT', 'count'),
    inst_late_count = ('is_late', 'sum'),
    inst_avg_diff   = ('payment_diff', 'mean'),
    inst_max_late   = ('days_late', 'max'),
)
inst_agg['inst_late_rate'] = inst_agg['inst_late_count'] / inst_agg['inst_count']

inst_agg.to_parquet(PROC / 'inst_agg.parquet')
print(f"inst_agg shape: {inst_agg.shape}")
del inst, inst_agg

inst_agg shape: (252579, 5)


## 5. Aggregate credit_card_balance.csv

In [6]:
cc = pd.read_csv(RAW / 'credit_card_balance.csv')

# Last 6 months only
cc = cc[cc['MONTHS_BALANCE'] >= -6]

cc_agg = cc.groupby('SK_ID_CURR').agg(
    cc_avg_balance = ('AMT_BALANCE', 'mean'),
    cc_avg_limit   = ('AMT_CREDIT_LIMIT_ACTUAL', 'mean'),
    cc_avg_payment = ('AMT_PAYMENT_CURRENT', 'mean'),
    cc_max_dpd     = ('SK_DPD', 'max'),
    cc_max_dpd_def = ('SK_DPD_DEF', 'max'),
)
cc_agg['cc_utilisation'] = cc_agg['cc_avg_balance'] / cc_agg['cc_avg_limit'].replace(0, pd.NA)

cc_agg.to_parquet(PROC / 'cc_agg.parquet')
print(f"cc_agg shape: {cc_agg.shape}")
del cc, cc_agg

cc_agg shape: (103556, 6)


## 6. Build master table

In [7]:
master = pd.read_parquet(PROC / 'application_slim.parquet')

for path in [
    PROC / 'bureau_agg.parquet',
    PROC / 'prev_agg.parquet',
    PROC / 'inst_agg.parquet',
    PROC / 'cc_agg.parquet',
]:
    master = master.merge(pd.read_parquet(path), on='SK_ID_CURR', how='left')

master.to_parquet(PROC / 'master.parquet', index=False)
print(f"Master shape: {master.shape}")
print(f"Default rate: {master['TARGET'].mean():.2%}")
print(f"\nMissing rates (top 10):\n{master.isnull().mean().sort_values(ascending=False).head(10).round(3).to_string()}")

Master shape: (307511, 61)
Default rate: 8.07%

Missing rates (top 10):
cc_avg_payment        0.801
cc_utilisation        0.779
cc_avg_limit          0.717
cc_avg_balance        0.717
cc_max_dpd            0.717
cc_max_dpd_def        0.717
OWN_CAR_AGE           0.660
EXT_SOURCE_1          0.564
bureau_max_overdue    0.402
OCCUPATION_TYPE       0.313
